# Week 4 — GNSS + IMU Fusion

This notebook introduces a simple 1D Kalman filter.
You are given:
- a GNSS position measurement
- IMU acceleration
- ground truth for evaluation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../datasets/week4_gnss_imu_fusion.csv")
df.head()

## State model

Use the state:
\[
x_k = \begin{bmatrix} p_k \\ v_k \end{bmatrix}
\]

with acceleration as a known input.

In [ ]:
dt = 1.0
A = np.array([[1, dt],
              [0, 1]])
B = np.array([[0.5 * dt**2],
              [dt]])
H = np.array([[1, 0]])

Q = np.array([[0.5, 0.0],
              [0.0, 0.5]])   # TODO: tune
R = np.array([[6.0]])        # TODO: tune

## Task 1: Implement the Kalman filter loop

In [ ]:
xhat = np.zeros((len(df), 2))
P = np.eye(2)

for k in range(1, len(df)):
    u = df.loc[k-1, "imu_acc_mps2"]

    # Predict
    x_pred = A @ xhat[k-1] + (B.flatten() * u)
    P_pred = A @ P @ A.T + Q

    # Update with GNSS
    z = np.array([[df.loc[k, "gnss_pos_meas_m"]]])

    # TODO: compute innovation, Kalman gain, posterior state, posterior covariance
    # y = z - H @ x_pred.reshape(-1,1)
    # S = ...
    # K = ...
    # x_post = ...
    # P = ...

    # Temporary placeholder so notebook runs once completed:
    # xhat[k] = x_post.flatten()

## Task 2: Plot performance

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(df["t_s"], df["true_x_m"], label="true position")
ax.plot(df["t_s"], df["gnss_pos_meas_m"], label="GNSS measurement", alpha=0.6)
# ax.plot(df["t_s"], xhat[:,0], label="fused estimate")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Position [m]")
ax.legend()
plt.show()

## Task 3: Quantify error before and during spoofing

In [ ]:
# TODO after filter works
# df["fused_error_m"] = xhat[:,0] - df["true_x_m"]
# df["gnss_error_m"] = df["gnss_pos_meas_m"] - df["true_x_m"]
# pre = df["spoof_active"] == 0
# post = df["spoof_active"] == 1
# {
#     "gnss_rmse_pre": np.sqrt(np.mean(df.loc[pre, "gnss_error_m"]**2)),
#     "gnss_rmse_post": np.sqrt(np.mean(df.loc[post, "gnss_error_m"]**2)),
#     "fused_rmse_pre": np.sqrt(np.mean(df.loc[pre, "fused_error_m"]**2)),
#     "fused_rmse_post": np.sqrt(np.mean(df.loc[post, "fused_error_m"]**2)),
# }